# Crop-marketing participation logistic regression

Reconstructed workflow for Table 4.15.

Personal file paths, stored outputs, exploratory analyses, and superseded cells have been removed.

## Verification required

The uploaded notebook does not retain the original code cell that generated Table 4.15.
This notebook reconstructs the specification described in the thesis and uses the earlier
training-variable treatment visible in the stored notebook state. It must not be presented
as the original result-producing code until its coefficients are checked against Table 4.15.

In [ ]:
from pathlib import Path
import warnings

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name in {"notebooks", "needs_verification"}:
    PROJECT_DIR = PROJECT_DIR.parent

DATA_FILE = PROJECT_DIR / "data" / "questionnaire.xlsx"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {DATA_FILE}\n"
        "Place the anonymised questionnaire file in data/questionnaire.xlsx "
        "or update DATA_FILE."
    )

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

OUT_DIR = OUTPUTS_DIR / "06_marketing_participation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(DATA_FILE)
df.columns = df.columns.str.strip()

gender_col = "What is your gender?"
location_col = "Location"
training_col = "If yes, how many training programs have you attended?"
adaptation_col = "Have you implemented at least one adaptation strategy?"
marketing_col = "Are women involved in decision-making processes related to crop selection, marketing, and income management?"

required = [gender_col, location_col, training_col, adaptation_col, marketing_col]
missing = [column for column in required if column not in df.columns]
if missing:
    raise KeyError(f"Missing questionnaire columns:\n{missing}")

df["female"] = (
    df[gender_col].astype("string").str.strip().str.lower().eq("female").astype(int)
)
df["PIS"] = (
    df[location_col].astype("string").str.contains("Perkerra", case=False, na=False).astype(int)
)

# Missing training responses are intentionally retained as missing here.
# This reflects the earlier variable state associated with the published
# highly unstable Table 4.15 model.
df["train_5plus"] = (
    df[training_col].astype("string").str.strip().str.lower()
      .map({"1-5": 0, "5-10": 1, ">10": 1, "more than 10": 1})
)

df["adopted_CSA"] = (
    df[adaptation_col].astype("string").str.strip().str.lower()
      .map({"yes": 1, "no": 0})
)

df["marketing_participation"] = (
    df[marketing_col].astype("string").str.strip().str.lower()
      .map({"yes": 1, "no": 0})
)

model_data = df[
    ["marketing_participation", "female", "PIS", "train_5plus", "adopted_CSA"]
].dropna().copy()

X = sm.add_constant(
    model_data[["female", "PIS", "train_5plus", "adopted_CSA"]],
    has_constant="add",
)
y = model_data["marketing_participation"]

model = sm.Logit(y, X).fit(disp=False, maxiter=200)
ci = model.conf_int()

table_415 = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": model.params.values,
    "StdErr": model.bse.values,
    "p_value": model.pvalues.values,
    "Odds_Ratio": np.exp(model.params.values),
    "CI_Low": np.exp(ci[0].values),
    "CI_High": np.exp(ci[1].values),
})

table_415["Variable"] = table_415["Variable"].replace({
    "const": "const",
    "train_5plus": "≥5 trainings",
    "adopted_CSA": "Adopted at least one CSA practice",
})
table_415.to_excel(OUT_DIR / "table_4_15_marketing_logit.xlsx", index=False)

display(table_415)
print(f"Model N = {int(model.nobs)}")
print("\nCompare these coefficients against the published Table 4.15 before upload.")
